# Chapter 2 · Diet Coach Agent Team

This notebook is organised as small, book-friendly snippets.

We first build a simple diet coach team with CrewAI, then rebuild the same workflow with plain OpenAI API calls.

## 2.0 · Setup

In [ ]:
# !pip install crewai crewai-tools openai python-dotenv

import json
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_MODEL = "gpt-4o-mini"

## 2.1 · Mock data

In [ ]:
FOOD_DB = {
    "greek yogurt": {"calories": 100, "protein_g": 17, "carbs_g": 6, "fat_g": 0.7},
    "oats": {"calories": 150, "protein_g": 5, "carbs_g": 27, "fat_g": 2.5},
    "banana": {"calories": 105, "protein_g": 1.3, "carbs_g": 27, "fat_g": 0.4},
    "chicken breast": {"calories": 165, "protein_g": 31, "carbs_g": 0, "fat_g": 3.6},
    "brown rice": {"calories": 216, "protein_g": 5, "carbs_g": 45, "fat_g": 1.8},
    "broccoli": {"calories": 55, "protein_g": 3.7, "carbs_g": 11, "fat_g": 0.6},
    "salmon": {"calories": 208, "protein_g": 28, "carbs_g": 0, "fat_g": 10},
    "spinach": {"calories": 23, "protein_g": 2.9, "carbs_g": 3.6, "fat_g": 0.4},
}

GOAL_TEMPLATES = {
    "fat loss": {
        "priority": "high protein, high fibre, controlled calories",
        "protein_target_g": 120,
    },
    "muscle gain": {
        "priority": "higher calories, high protein, balanced carbs",
        "protein_target_g": 150,
    },
}

MEAL_PLAN_LOG = []

## 2.2 · Tool functions

In [ ]:
def lookup_food(food_name: str) -> str:
    key = food_name.strip().lower()
    if key in FOOD_DB:
        return json.dumps({"food": key, **FOOD_DB[key]})
    return json.dumps({"error": f"'{food_name}' is not in the food database"})


def get_goal_template(goal: str) -> str:
    key = goal.strip().lower()
    if key in GOAL_TEMPLATES:
        return json.dumps({"goal": key, **GOAL_TEMPLATES[key]})
    return json.dumps({"error": f"Unknown goal '{goal}'"})


def save_meal_plan(user_name: str, plan_markdown: str) -> str:
    MEAL_PLAN_LOG.append({"user_name": user_name, "plan": plan_markdown})
    return json.dumps({"saved": True, "plans_saved": len(MEAL_PLAN_LOG)})

In [ ]:
CLIENT_PROFILE = {
    "name": "Maya",
    "goal": "fat loss",
    "dietary_preferences": "high-protein, mostly whole foods",
    "dislikes": "mushrooms",
    "calories_target": 1800,
    "meals_per_day": 3,
    "candidate_foods": [
        "greek yogurt",
        "oats",
        "banana",
        "chicken breast",
        "broccoli",
        "brown rice",
        "salmon",
        "spinach",
    ],
}

print(lookup_food("salmon"))
print(get_goal_template("fat loss"))

## 2.3 · CrewAI imports

In [ ]:
from crewai import Agent, Crew, LLM, Process, Task
from crewai_tools import tool

## 2.4 · Registering tools for CrewAI

In [ ]:
llm = LLM(model=OPENAI_MODEL, temperature=0.2)

@tool("goal_template_lookup")
def goal_template_lookup(goal: str) -> str:
    """Look up the nutrition goal template."""
    return get_goal_template(goal)


@tool("food_lookup")
def food_lookup(food_name: str) -> str:
    """Look up nutrition facts for one food."""
    return lookup_food(food_name)


@tool("plan_saver")
def plan_saver(user_name: str, plan_markdown: str) -> str:
    """Save the final meal plan."""
    return save_meal_plan(user_name, plan_markdown)

## 2.5 · Defining the agents

In [ ]:
intake_agent = Agent(
    role="Intake Coach",
    goal="Turn the client brief into clear diet requirements.",
    backstory="You clarify the user's goal, calorie target, and food constraints.",
    tools=[goal_template_lookup],
    llm=llm,
    verbose=True,
)

analyst_agent = Agent(
    role="Nutrition Analyst",
    goal="Choose foods that fit the client's goal.",
    backstory="You use calories and protein data to select sensible foods.",
    tools=[food_lookup],
    llm=llm,
    verbose=True,
)

planner_agent = Agent(
    role="Meal Planner",
    goal="Create a simple one-day meal plan.",
    backstory="You turn the brief and food analysis into practical meals.",
    tools=[plan_saver],
    llm=llm,
    verbose=True,
)

## 2.6 · Writing the tasks

In [ ]:
intake_task = Task(
    description=(
        "The client is {name}. Goal: {goal}. Preferences: {dietary_preferences}. "
        "Dislikes: {dislikes}. Calories target: {calories_target}. Meals per day: {meals_per_day}. "
        "Use the goal template tool and write a short intake brief."
    ),
    expected_output="A short intake brief with priorities and protein target.",
    agent=intake_agent,
)

analyst_task = Task(
    description=(
        "Using the intake brief and these candidate foods: {candidate_foods}, choose the best foods. "
        "Use the food lookup tool and return 5-7 foods with a short reason for each."
    ),
    expected_output="A shortlist of foods with short reasons.",
    agent=analyst_agent,
)

planner_task = Task(
    description=(
        "Create a one-day meal plan for {name}. Write breakfast, lunch, dinner, and one coaching note. "
        "Keep it close to {calories_target} calories. Then save the plan."
    ),
    expected_output="A saved one-day meal plan.",
    agent=planner_agent,
)

## 2.7 · Orchestrating the crew

In [ ]:
crew = Crew(
    agents=[intake_agent, analyst_agent, planner_agent],
    tasks=[intake_task, analyst_task, planner_task],
    process=Process.sequential,
    memory=True,
    verbose=True,
)

result = crew.kickoff(
    inputs={
        "name": CLIENT_PROFILE["name"],
        "goal": CLIENT_PROFILE["goal"],
        "dietary_preferences": CLIENT_PROFILE["dietary_preferences"],
        "dislikes": CLIENT_PROFILE["dislikes"],
        "calories_target": CLIENT_PROFILE["calories_target"],
        "meals_per_day": CLIENT_PROFILE["meals_per_day"],
        "candidate_foods": ", ".join(CLIENT_PROFILE["candidate_foods"]),
    }
)

print(result)

## 2.8 · Plain OpenAI version

In [ ]:
from openai import OpenAI

# OpenAI is the default:
# OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# DeepSeek:
# OpenAI(api_key=os.environ.get("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
# Gemini:
# OpenAI(api_key=os.environ.get("GEMINI_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
# Claude via an OpenAI-compatible provider such as OpenRouter:
# OpenAI(api_key=os.environ.get("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
client = OpenAI()

## 2.9 · Step 1: interpret the client brief

In [ ]:
goal_template = get_goal_template(CLIENT_PROFILE["goal"])

planning_brief = client.chat.completions.create(
    model=OPENAI_MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a concise diet coach."},
        {
            "role": "user",
            "content": f"""
Interpret this client brief and write a short planning brief.

Client:
{json.dumps(CLIENT_PROFILE, indent=2)}

Goal template:
{goal_template}
""",
        },
    ],
).choices[0].message.content

print(planning_brief)

## 2.10 · Step 2: choose foods

In [ ]:
food_facts = [json.loads(lookup_food(food)) for food in CLIENT_PROFILE["candidate_foods"]]

food_shortlist = client.chat.completions.create(
    model=OPENAI_MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a nutrition analyst. Use only the provided food facts."},
        {
            "role": "user",
            "content": f"""
Planning brief:
{planning_brief}

Food facts:
{json.dumps(food_facts, indent=2)}

Choose the best 5-7 foods and explain each choice in one short sentence.
""",
        },
    ],
).choices[0].message.content

print(food_shortlist)

## 2.11 · Step 3: create the meal plan

In [ ]:
meal_plan = client.chat.completions.create(
    model=OPENAI_MODEL,
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a practical diet coach writing simple meal plans."},
        {
            "role": "user",
            "content": f"""
Create a one-day meal plan for this client.

Client:
{json.dumps(CLIENT_PROFILE, indent=2)}

Planning brief:
{planning_brief}

Food shortlist:
{food_shortlist}

Write breakfast, lunch, dinner, one snack, and one coaching note.
""",
        },
    ],
).choices[0].message.content

save_meal_plan(CLIENT_PROFILE["name"], meal_plan)
print(meal_plan)